# Activation Similarity

For each transformer layer `l`, compare the layer input at generated step `t + 1` against the same layer's output at generated step `t` while `Qwen/Qwen3.5-0.8B` generates text.

In [ ]:
%pip install -U pip setuptools wheel
%pip install -U torch accelerate safetensors sentencepiece protobuf huggingface_hub hf_transfer
%pip install -U "transformers[torch] @ git+https://github.com/huggingface/transformers.git@main"

: 

In [ ]:
import os
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3.5-0.8B"
PROMPT = "Write a short explanation of why hidden-state similarity changes during generation."
MAX_NEW_TOKENS = 500

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.bfloat16 if device in {"cuda", "mps"} else torch.float32
device, dtype

In [ ]:
def read_env_value(path, key):
    for line in path.read_text(errors="ignore").splitlines():
        name, sep, value = line.strip().partition("=")
        if sep and name == key:
            return value.strip().strip('"').strip("'")


def find_hf_token():
    paths = sorted(Path.home().joinpath("Projects").rglob(".env.local"))
    token = os.environ.get("HF_TOKEN") or next((read_env_value(path, "HF_TOKEN") for path in paths if read_env_value(path, "HF_TOKEN")), None)
    assert token, "No HF_TOKEN found in environment or ~/Projects/**/.env.local"
    os.environ["HF_TOKEN"] = token
    return token

HF_TOKEN = find_hf_token()
print("HF_TOKEN loaded")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    token=HF_TOKEN,
    dtype=dtype,
    device_map="auto" if device == "cuda" else None,
    trust_remote_code=True,
).eval()
model = model if device == "cuda" else model.to(device)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
assert tokenizer.pad_token_id is not None
assert model.config.num_hidden_layers > 0

In [ ]:
@torch.no_grad()
def generate_sequence(prompt, max_new_tokens=MAX_NEW_TOKENS):
    batch = tokenizer(prompt, return_tensors="pt").to(model.device)
    seq = model.generate(
        **batch,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    assert seq.shape[1] > batch.input_ids.shape[1]
    return tokenizer.decode(seq[0], skip_special_tokens=True), seq, batch.input_ids.shape[1]

text, sequence, prompt_len = generate_sequence(PROMPT)
print(text)

In [ ]:
@torch.no_grad()
def generated_token_states(sequence, prompt_len):
    out = model(sequence.to(model.device), output_hidden_states=True, use_cache=False)
    states = torch.stack([h[0, prompt_len:, :].float().cpu() for h in out.hidden_states], dim=1)
    assert states.ndim == 3, states.shape
    assert states.shape[0] >= 2, "Need at least 2 generated tokens for t vs t+1 similarity"
    assert states.shape[1] == model.config.num_hidden_layers + 1
    return states

states = generated_token_states(sequence, prompt_len)
states.shape  # (generated_tokens, embedding_plus_layer_outputs, hidden_dim)

In [ ]:
def layer_input_next_vs_output_now_scores(states):
    layer_inputs_next = states[1:, :-1]
    layer_outputs_now = states[:-1, 1:]
    sims = F.cosine_similarity(layer_inputs_next, layer_outputs_now, dim=-1)
    assert sims.shape == (states.shape[0] - 1, model.config.num_hidden_layers)
    means = sims.mean(dim=0)
    return sims, means

similarities, layer_scores = layer_input_next_vs_output_now_scores(states)
for layer, score in enumerate(layer_scores.tolist()):
    print(f"layer_{layer:02d}: {score:.4f}")

In [ ]:
# Tiny sanity tests for the scoring tensors.
assert states.shape[0] == sequence.shape[1] - prompt_len
assert similarities.shape == (states.shape[0] - 1, model.config.num_hidden_layers)
assert torch.isfinite(similarities).all()
assert torch.all(similarities <= 1.0001)
assert torch.all(similarities >= -1.0001)
print("tests passed")